# Dummy Agent Library

In this simple example, **we're going to code an Agent from scratch**.

This notebook is part of the <a href="https://www.hf.co/learn/agents-course">Hugging Face Agents Course</a>, a free Course from beginner to expert, where you learn to build Agents.

<img src="https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png" alt="Agent Course"/>

In [1]:
!pip install -q huggingface_hub

In [2]:
!pip install -q google-genai

In [3]:
import os
from google import genai
from google.colab import userdata

# Intentamos obtener la clave de tus secretos de Colab
try:
    GEMINI_API_KEY = userdata.get("GEMINI_TOKEN") or userdata.get("GOOGLE_TOKEN")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_TOKEN") or os.environ.get("GOOGLE_TOKEN")

if not GEMINI_API_KEY:
    raise ValueError("Por favor, asegúrate de añadir tu clave de API en la pestaña 'Secrets' (icono de llave a la izquierda) con el nombre 'GEMINI_API_KEY'.")

# Inicializamos el cliente de Gemini para usarlo en lugar del cliente de Hugging Face
client_gemini = genai.Client(api_key=GEMINI_API_KEY)

# Creamos un adaptador simple para que funcione de manera idéntica al client.chat.completions.create original de Hugging Face / OpenAI
class GeminiAdapter:
    def __init__(self, client):
        self.client = client
        self.chat = self
        self.completions = self

    def create(self, messages, model="gemini-3.8-flash", stream=False, max_tokens=1024, stop=None, **kwargs):
        # Convertimos el formato de mensajes de OpenAI a contenido de Gemini
        contents = []
        system_instruction = None

        for m in messages:
            role = m["role"]
            content = m["content"]
            if role == "system":
                system_instruction = content
            else:
                # Mapeamos 'assistant' a 'model'
                gemini_role = "model" if role == "assistant" else "user"
                contents.append({"role": gemini_role, "parts": [{"text": content}]})

        # Configuración de generación
        config_args = {}
        if max_tokens:
            config_args["max_output_tokens"] = max_tokens
        if stop:
            config_args["stop_sequences"] = stop
        if system_instruction:
            config_args["system_instruction"] = system_instruction

        config = genai.types.GenerateContentConfig(**config_args) if config_args else None

        response = self.client.models.generate_content(
            model=model,
            contents=contents,
            config=config
        )

        # Mock para mantener compatibilidad con structure.choices[0].message.content
        class ChoiceMessage:
            def __init__(self, text):
                self.content = text
        class Choice:
            def __init__(self, text):
                self.message = ChoiceMessage(text)
        class ResponseMock:
            def __init__(self, text):
                self.choices = [Choice(text)]

        return ResponseMock(response.text)

# Sobrescribimos 'client' para que el resto del código del notebook funcione sin cambiar nada más
client = GeminiAdapter(client_gemini)
print("¡Cliente de Gemini (gemini-3.8-flash) configurado con éxito como reemplazo de Hugging Face!")

¡Cliente de Gemini (gemini-3.8-flash) configurado con éxito como reemplazo de Hugging Face!


Ahora que hemos configurado este adaptador de Gemini, todas las llamadas a `client.chat.completions.create` del notebook funcionarán utilizando los modelos de Google (como `gemini-2.5-flash`) de manera transparente y 100% gratuita.

## Serverless API

In the Hugging Face ecosystem, there is a convenient feature called Serverless API that allows you to easily run inference on many models. There's no installation or deployment required.

To run this notebook, **you need a Hugging Face token** that you can get from https://hf.co/settings/tokens. A "Read" token type is sufficient.
- If you are running this notebook on Google Colab, you can set it up in the "settings" tab under "secrets". Make sure to call it "HF_TOKEN" and restart the session to load the environment variable (Runtime -> Restart session).
- If you are running this notebook locally, you can set it up as an [environment variable](https://huggingface.co/docs/huggingface_hub/en/package_reference/environment_variables). Make sure you restart the kernel after installing or updating huggingface_hub. You can update huggingface_hub by modifying the above `!pip install -q huggingface_hub -U`

In [4]:
import os
from huggingface_hub import InferenceClient

## You need a token from https://hf.co/settings/tokens, ensure that you select 'read' as the token type.
## If you run this on Google Colab, add it in the "Secrets" tab (key icon on the left sidebar) and call it "HF_TOKEN".
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except ImportError:
    HF_TOKEN = os.environ.get("HF_TOKEN")

client = InferenceClient(model="moonshotai/Kimi-K2.5", token=HF_TOKEN)

We use the `chat` method since it is a convenient and reliable way to apply chat templates:

In [7]:
# Restauramos client al adaptador de Gemini definido en la celda anterior para evitar usar la API de pago de Hugging Face
client = GeminiAdapter(client_gemini)

try:
    output = client.chat.completions.create(
        model="gemini-3.8-flash",
        messages=[
            {"role": "user", "content": "The capital of France is"},
        ],
        stream=False,
        max_tokens=1024,
    )
except Exception as e:
    print("gemini-3.8-flash está saturado, probando con modelo alternativo...")
    output = client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[
            {"role": "user", "content": "The capital of France is"},
        ],
        stream=False,
        max_tokens=1024,
    )

print(output.choices[0].message.content)

Paris.


The chat method is the RECOMMENDED method to use in order to ensure a **smooth transition between models**.

## Dummy Agent

In the previous sections, we saw that the **core of an agent library is to append information in the system prompt**.

This system prompt is a bit more complex than the one we saw earlier, but it already contains:

1. **Information about the tools**
2. **Cycle instructions** (Thought → Action → Observation)

In [8]:
# This system prompt is a bit more complex and actually contains the function description already appended.
# Here we suppose that the textual description of the tools has already been appended.

SYSTEM_PROMPT = """Answer the following questions as best you can. You have access to the following tools:

get_weather: Get the current weather in a given location

The way you use the tools is by specifying a json blob.
Specifically, this json should have an `action` key (with the name of the tool to use) and an `action_input` key (with the input to the tool going here).

The only values that should be in the "action" field are:
get_weather: Get the current weather in a given location, args: {"location": {"type": "string"}}
example use :

{{
  "action": "get_weather",
  "action_input": {{"location": "New York"}}
}}


ALWAYS use the following format:

Question: the input question you must answer
Thought: you should always think about one action to take. Only one action at a time in this format:
Action:

$JSON_BLOB (inside markdown cell)

Observation: the result of the action. This Observation is unique, complete, and the source of truth.
... (this Thought/Action/Observation can repeat N times, you should take several steps when needed. The $JSON_BLOB must be formatted as markdown and only use a SINGLE action at a time.)

You must always end your output with the following format:

Thought: I now know the final answer
Final Answer: the final answer to the original input question

Now begin! Reminder to ALWAYS use the exact characters `Final Answer:` when you provide a definitive answer. """

We need to append the user instruction after the system prompt. This happens inside the `chat` method. We can see this process below:

In [9]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What's the weather in London?"},
]

print(messages)

[{'role': 'system', 'content': 'Answer the following questions as best you can. You have access to the following tools:\n\nget_weather: Get the current weather in a given location\n\nThe way you use the tools is by specifying a json blob.\nSpecifically, this json should have an `action` key (with the name of the tool to use) and an `action_input` key (with the input to the tool going here).\n\nThe only values that should be in the "action" field are:\nget_weather: Get the current weather in a given location, args: {"location": {"type": "string"}}\nexample use :\n\n{{\n  "action": "get_weather",\n  "action_input": {{"location": "New York"}}\n}}\n\n\nALWAYS use the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about one action to take. Only one action at a time in this format:\nAction:\n\n$JSON_BLOB (inside markdown cell)\n\nObservation: the result of the action. This Observation is unique, complete, and the source of truth.\n... (thi

Let's call the `chat` method!

In [11]:
try:
    output = client.chat.completions.create(
        model="gemini-3.8-flash",
        messages=messages,
        stream=False,
        max_tokens=200,
        extra_body={'thinking': {'type': 'disabled'}},
    )
except Exception as e:
    print("gemini-3.8-flash está saturado, probando con modelo alternativo...")
    output = client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=messages,
        stream=False,
        max_tokens=200,
        extra_body={'thinking': {'type': 'disabled'}},
    )

print(output.choices[0].message.content)

Thought: I need to check the current weather in London.
Action:
```json
{
  "action": "get_weather",
  "action_input": {
    "location": "London"
  }
}
```


Do you see the issue?

> At this point, the model is hallucinating, because it's producing a fabricated "Observation" -- a response that it generates on its own rather than being the result of an actual function or tool call.
> To prevent this, we stop generating right before "Observation:".
> This allows us to manually run the function (e.g., `get_weather`) and then insert the real output as the Observation.

In [12]:
# The answer was hallucinated by the model. We need to stop to actually execute the function!
try:
    output = client.chat.completions.create(
        model="gemini-3.8-flash",
        messages=messages,
        max_tokens=150,
        stop=["Observation:"], # Let's stop before any actual function is called
        extra_body={'thinking': {'type': 'disabled'}},
    )
except Exception as e:
    print("gemini-3.8-flash está saturado, probando con modelo alternativo...")
    output = client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=messages,
        max_tokens=150,
        stop=["Observation:"], # Let's stop before any actual function is called
        extra_body={'thinking': {'type': 'disabled'}},
    )

print(output.choices[0].message.content)

Thought: I need to check the current weather in London.
Action:
```json
{
  "action": "get_weather",
  "action_input": {
    "location": "London"
  }
}
```


Much Better!

Let's now create a **dummy get weather function**. In a real situation you could call an API.

In [13]:
# Dummy function
def get_weather(location):
    return f"the weather in {location} is sunny with low temperatures. \n"

get_weather('London')

'the weather in London is sunny with low temperatures. \n'

Let's concatenate the system prompt, the base prompt, the completion until function execution and the result of the function as an Observation and resume generation.

In [14]:
# Para adaptarnos a las reglas de Gemini, enviamos la observación simulando ser un mensaje del usuario
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What's the weather in London?"},
    {"role": "assistant", "content": output.choices[0].message.content},
    {"role": "user", "content": "Observation:\n" + get_weather('London')}
]

try:
    output = client.chat.completions.create(
        model="gemini-3.8-flash",
        messages=messages,
        stream=False,
        max_tokens=200,
        extra_body={'thinking': {'type': 'disabled'}},
    )
except Exception as e:
    print("gemini-3.8-flash está saturado, probando con modelo alternativo...")
    output = client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=messages,
        stream=False,
        max_tokens=200,
        extra_body={'thinking': {'type': 'disabled'}},
    )

print(output.choices[0].message.content)

Thought: I now know the final answer
Final Answer: The weather in London is currently sunny with low temperatures.


We learned how we can create Agents from scratch using Python code, and we **saw just how tedious that process can be**. Fortunately, many Agent libraries simplify this work by handling much of the heavy lifting for you.

Now, we're ready **to create our first real Agent** using the `smolagents` library.